# DeepGuard AI - Notebook 01: Dataset Exploration & Leakage-Free Splitting
### Project: Deepfake Image & Video Detection System
**Objective:** Inspect and structure the FaceForensics++ (FF++) and Celeb-DF v2 datasets.
Learn how real and manipulated video sequences are organized and enforce strict identity-based grouping to prevent data leakage.

### Step 1: Environment Setup in Google Colab
Mount Google Drive where your dataset archive or downloaded folders reside, and clone or copy the project training source code.

In [ ]:
# Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

# Install dependencies
!pip install timm opencv-python-headless pandas scikit-learn matplotlib
import os
import glob
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

### Step 2: Understand FaceForensics++ Directory Structure
FaceForensics++ is composed of:
1. `original_sequences/youtube/c23/videos/` -> Pristine real videos
2. `manipulated_sequences/<method>/c23/videos/` -> Manipulated videos:
   - `Deepfakes`: Classic autoencoder face-swapping
   - `Face2Face`: Facial reenactment
   - `FaceSwap`: Graphic-based face swapping
   - `NeuralTextures`: GAN-based neural texture rendering

**Compression Levels:** `c23` (H.264 high quality) is recommended for training.

In [ ]:
# Configure dataset root path on your Google Drive
DATASET_ROOT = "/content/drive/MyDrive/FaceForensics++"

print("Checking dataset root:", DATASET_ROOT)
if os.path.exists(DATASET_ROOT):
    print("Contents:", os.listdir(DATASET_ROOT))
else:
    print("[NOTE] Path does not exist yet. Please configure DATASET_ROOT to your FF++ folder location.")

### Step 3: Preventing Data Leakage
In deepfake research, an actor's face appears across multiple manipulated videos.
For example, original video `042` generates manipulated videos `042_056`, `042_089`, etc.
**If video 042 is in the training set while 042_056 is in the validation set, the network memorizes the actor's facial identity rather than learning manipulation artifacts.**

To prevent this, we group strictly by `source_id`.

In [ ]:
def parse_ff_directory(dataset_root, compression="c23"):
    methods = ["Deepfakes", "Face2Face", "FaceSwap", "NeuralTextures"]
    records = []

    # Real videos
    real_paths = glob.glob(os.path.join(dataset_root, "original_sequences/youtube", compression, "videos/*.mp4"))
    for p in real_paths:
        base = os.path.splitext(os.path.basename(p))[0]
        records.append({"video_path": p, "label": 0, "class_name": "REAL", "method": "original", "source_id": base})

    # Manipulated videos
    for m in methods:
        fake_paths = glob.glob(os.path.join(dataset_root, "manipulated_sequences", m, compression, "videos/*.mp4"))
        for p in fake_paths:
            base = os.path.splitext(os.path.basename(p))[0]
            src = base.split("_")[0]
            records.append({"video_path": p, "label": 1, "class_name": "FAKE", "method": m, "source_id": src})

    return pd.DataFrame(records)

# Example demonstration with synthetic verification
print("Manifest builder defined. Enforces strict source_id isolation.")

### Step 4: Split Generation (Train 70% / Val 15% / Test 15%)
We split unique `source_id`s deterministically using a fixed random seed (42).

In [ ]:
import random

def split_by_identity(df, seed=42, train_ratio=0.7, val_ratio=0.15):
    sources = sorted(df["source_id"].unique())
    rng = random.Random(seed)
    rng.shuffle(sources)

    n = len(sources)
    n_train = int(n * train_ratio)
    n_val = int(n * val_ratio)

    train_src = set(sources[:n_train])
    val_src = set(sources[n_train:n_train+n_val])
    test_src = set(sources[n_train+n_val:])

    train_df = df[df["source_id"].isin(train_src)].copy()
    val_df = df[df["source_id"].isin(val_src)].copy()
    test_df = df[df["source_id"].isin(test_src)].copy()

    # Leakage assertion check
    assert len(train_src.intersection(val_src)) == 0, "CRITICAL ERROR: Leakage between train and val!"
    assert len(train_src.intersection(test_src)) == 0, "CRITICAL ERROR: Leakage between train and test!"
    print(f"Splits generated successfully with 0 identity leakage:")
    print(f"  Train: {len(train_df)} videos ({len(train_src)} identities)")
    print(f"  Val:   {len(val_df)} videos ({len(val_src)} identities)")
    print(f"  Test:  {len(test_df)} videos ({len(test_src)} identities)")
    return train_df, val_df, test_df

### Next Steps:
Now proceed to `02_video_preprocessing.ipynb` to sample video frames and extract face crops.